# Prerequisite Violation Report

This notebook identifies participants who registered for a course without completing all prerequisite courses before the enrollment date.

The report uses the following schema from the SQLite database:

- `courses`: course metadata and prerequisite list
- `enrollments`: enrollment records for participants

In `courses`, `prerequisites` is stored as a JSON array of course ids, e.g. `[1,2,3]`.

In [2]:
import sqlite3
import pandas as pd

db_path = 'data/courses.db'
conn = sqlite3.connect(db_path)
query = '''
SELECT
    e.enrollment_id,
    e.participant_id,
    e.participant_name,
    e.course_id,
    e.course_date,
    c.prerequisites
FROM enrollments AS e
JOIN courses AS c
    ON c.course_id = e.course_id
WHERE EXISTS (
    SELECT 1
    FROM json_each(c.prerequisites) AS prereq
    WHERE NOT EXISTS (
        SELECT 1
        FROM enrollments AS done
        WHERE done.participant_id = e.participant_id
          AND done.course_id = prereq.value
          AND done.course_date < e.course_date
    )
);
'''
df = pd.read_sql_query(query, conn)
conn.close()
df

,enrollment_id,participant_id,participant_name,course_id,course_date,prerequisites
0,10,P002,Bob Lee,3,2024-03-05,"[1,2]"
1,11,P005,Edward Goh,4,2024-03-15,[3]
2,13,P008,Henry Teo,3,2024-03-25,"[1,2]"
3,21,P005,Edward Goh,5,None,[4]
4,26,P001,Alice Tan,4,1900-06-10,[3]
5,28,P009,Irene Chua,5,2024-06-20,[4]
